# 🚀 ETF Backtesting System - Complete Setup & Run

**Notebook นี้จะทำทุกอย่างให้คุณอัตโนมัติ!**

## 📋 สิ่งที่ Notebook นี้จะทำ:
1. ✅ ตรวจสอบว่า MySQL พร้อมใช้งาน
2. ✅ ติดตั้ง dependencies ที่จำเป็น
3. ✅ สร้าง database และ tables
4. ✅ เพิ่มข้อมูล ETF ตัวอย่าง 40 ตัว
5. ✅ ดึงข้อมูลราคาย้อนหลัง (2009-2025)
6. ✅ แสดงตัวอย่างการใช้งาน
7. ✅ Run Analytics ทั้ง 3 insights

---

## 🎯 วิธีใช้งาน:
**กด Shift + Enter ทีละ cell ตามลำดับ**

หรือ **Cell → Run All** (Run ทั้งหมดทีเดียว - แนะนำถ้าคุณมั่นใจว่า MySQL เปิดอยู่)

⚠️ **สำคัญ:** MySQL ต้องเปิดอยู่ก่อน! (XAMPP, MySQL Workbench, หรือ systemctl start mysql)

---

## Step 1: Import Libraries และตรวจสอบระบบ

In [ ]:
# Import standard libraries
import sys
import os
from pathlib import Path
import subprocess

# Project directory
PROJECT_DIR = Path(os.getcwd()).absolute()
print(f"✓ Project Directory: {PROJECT_DIR}")

# Add module paths
sys.path.insert(0, str(PROJECT_DIR / 'crud_operations'))
sys.path.insert(0, str(PROJECT_DIR / 'backtesting'))
sys.path.insert(0, str(PROJECT_DIR / 'analytics'))
sys.path.insert(0, str(PROJECT_DIR / 'data_collection'))

print("✓ Module paths configured")

## Step 2: ติดตั้ง Dependencies (ถ้ายังไม่มี)

In [ ]:
# ตรวจสอบและติดตั้ง dependencies
required_packages = [
    'mysql-connector-python',
    'pandas',
    'numpy',
    'yfinance',
    'matplotlib',
    'seaborn',
    'tabulate',
    'tqdm'
]

print("Checking dependencies...\n")
missing_packages = []

for package in required_packages:
    try:
        if package == 'mysql-connector-python':
            __import__('mysql.connector')
        else:
            __import__(package)
        print(f"✓ {package}")
    except ImportError:
        print(f"✗ {package} - missing")
        missing_packages.append(package)

if missing_packages:
    print(f"\nInstalling missing packages: {', '.join(missing_packages)}")
    for package in missing_packages:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
    print("\n✓ All dependencies installed!")
else:
    print("\n✓ All dependencies already installed!")

## Step 3: Database Configuration

**ใส่ MySQL credentials ของคุณที่นี่**

In [ ]:
# MySQL Configuration
DB_CONFIG = {
    'host': '127.0.0.1',
    'user': 'root',
    'password': 'krittanut123456',  # ⚠️ เปลี่ยนตรงนี้ถ้า password คุณไม่ตรง
    'database': 'etf_backtesting',
    'port': 3306
}

print("✓ Database configuration loaded")
print(f"  Host: {DB_CONFIG['host']}:{DB_CONFIG['port']}")
print(f"  User: {DB_CONFIG['user']}")
print(f"  Database: {DB_CONFIG['database']}")

## Step 4: ทดสอบการเชื่อมต่อ MySQL

In [ ]:
import mysql.connector
from mysql.connector import Error

def test_mysql_connection():
    """ทดสอบการเชื่อมต่อ MySQL"""
    try:
        # ลองเชื่อมต่อแบบไม่ระบุ database ก่อน
        conn = mysql.connector.connect(
            host=DB_CONFIG['host'],
            port=DB_CONFIG['port'],
            user=DB_CONFIG['user'],
            password=DB_CONFIG['password']
        )
        
        cursor = conn.cursor()
        cursor.execute("SELECT VERSION()")
        version = cursor.fetchone()[0]
        
        cursor.close()
        conn.close()
        
        print(f"✓ MySQL Connection Successful!")
        print(f"✓ MySQL Version: {version}")
        return True
        
    except Error as e:
        print(f"✗ MySQL Connection Failed!")
        print(f"  Error: {e}")
        print("\n💡 แนะนำ:")
        print("  1. ตรวจสอบว่า MySQL เปิดอยู่ (XAMPP หรือ MySQL service)")
        print("  2. ตรวจสอบ password ใน DB_CONFIG ด้านบน")
        print("  3. ลองรัน: mysql -u root -p")
        return False

# ทดสอบการเชื่อมต่อ
if test_mysql_connection():
    print("\n🎉 พร้อมดำเนินการต่อ!")
else:
    print("\n⚠️ กรุณาแก้ไขปัญหาการเชื่อมต่อก่อนดำเนินการต่อ")

## Step 5: สร้าง Database และ Tables

In [ ]:
def create_database_and_tables():
    """สร้าง database และ tables ทั้งหมด"""
    
    # SQL สำหรับสร้าง database
    create_db_sql = f"CREATE DATABASE IF NOT EXISTS {DB_CONFIG['database']} CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"
    
    # SQL สำหรับสร้าง tables
    table_sqls = [
        # ETFs table
        """
        CREATE TABLE IF NOT EXISTS etfs (
            ticker VARCHAR(10) PRIMARY KEY,
            name VARCHAR(255) NOT NULL,
            category VARCHAR(100),
            expense_ratio DECIMAL(5, 4),
            inception_date DATE,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
        """,
        
        # Daily Prices table
        """
        CREATE TABLE IF NOT EXISTS daily_prices (
            price_id INT AUTO_INCREMENT PRIMARY KEY,
            ticker VARCHAR(10) NOT NULL,
            date DATE NOT NULL,
            open DECIMAL(12, 4),
            high DECIMAL(12, 4),
            low DECIMAL(12, 4),
            close DECIMAL(12, 4),
            volume BIGINT,
            adjusted_close DECIMAL(12, 4),
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (ticker) REFERENCES etfs(ticker) ON DELETE CASCADE,
            UNIQUE KEY unique_ticker_date (ticker, date),
            INDEX idx_ticker_date (ticker, date)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
        """,
        
        # Portfolios table
        """
        CREATE TABLE IF NOT EXISTS portfolios (
            portfolio_id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(255) NOT NULL,
            description TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
        """,
        
        # Portfolio ETFs table
        """
        CREATE TABLE IF NOT EXISTS portfolio_etfs (
            portfolio_id INT,
            ticker VARCHAR(10),
            weight DECIMAL(5, 2) NOT NULL,
            PRIMARY KEY (portfolio_id, ticker),
            FOREIGN KEY (portfolio_id) REFERENCES portfolios(portfolio_id) ON DELETE CASCADE,
            FOREIGN KEY (ticker) REFERENCES etfs(ticker) ON DELETE CASCADE,
            CHECK (weight > 0 AND weight <= 100)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
        """,
        
        # Backtests table
        """
        CREATE TABLE IF NOT EXISTS backtests (
            backtest_id INT AUTO_INCREMENT PRIMARY KEY,
            portfolio_id INT NOT NULL,
            strategy_type VARCHAR(50) NOT NULL,
            start_date DATE NOT NULL,
            end_date DATE NOT NULL,
            initial_capital DECIMAL(15, 2) NOT NULL,
            final_value DECIMAL(15, 2),
            total_return DECIMAL(10, 4),
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (portfolio_id) REFERENCES portfolios(portfolio_id) ON DELETE CASCADE,
            INDEX idx_portfolio_strategy (portfolio_id, strategy_type)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
        """,
        
        # Backtest Transactions table
        """
        CREATE TABLE IF NOT EXISTS backtest_transactions (
            transaction_id INT AUTO_INCREMENT PRIMARY KEY,
            backtest_id INT NOT NULL,
            date DATE NOT NULL,
            ticker VARCHAR(10) NOT NULL,
            transaction_type ENUM('buy', 'sell') NOT NULL,
            shares DECIMAL(12, 6) NOT NULL,
            price DECIMAL(12, 4) NOT NULL,
            transaction_cost DECIMAL(12, 4),
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (backtest_id) REFERENCES backtests(backtest_id) ON DELETE CASCADE,
            INDEX idx_backtest_date (backtest_id, date)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
        """,
        
        # Backtest Portfolio Values table
        """
        CREATE TABLE IF NOT EXISTS backtest_portfolio_values (
            value_id INT AUTO_INCREMENT PRIMARY KEY,
            backtest_id INT NOT NULL,
            date DATE NOT NULL,
            portfolio_value DECIMAL(15, 2) NOT NULL,
            cash_balance DECIMAL(15, 2),
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (backtest_id) REFERENCES backtests(backtest_id) ON DELETE CASCADE,
            UNIQUE KEY unique_backtest_date (backtest_id, date),
            INDEX idx_backtest_date_value (backtest_id, date)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
        """,
        
        # Backtest Metrics table
        """
        CREATE TABLE IF NOT EXISTS backtest_metrics (
            metric_id INT AUTO_INCREMENT PRIMARY KEY,
            backtest_id INT NOT NULL,
            metric_name VARCHAR(100) NOT NULL,
            metric_value DECIMAL(15, 6),
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (backtest_id) REFERENCES backtests(backtest_id) ON DELETE CASCADE,
            UNIQUE KEY unique_backtest_metric (backtest_id, metric_name)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
        """
    ]
    
    try:
        # เชื่อมต่อ MySQL แบบไม่ระบุ database
        conn = mysql.connector.connect(
            host=DB_CONFIG['host'],
            port=DB_CONFIG['port'],
            user=DB_CONFIG['user'],
            password=DB_CONFIG['password']
        )
        cursor = conn.cursor()
        
        # สร้าง database
        print(f"Creating database '{DB_CONFIG['database']}'...")
        cursor.execute(create_db_sql)
        print(f"✓ Database '{DB_CONFIG['database']}' created/verified")
        
        # ใช้ database
        cursor.execute(f"USE {DB_CONFIG['database']}")
        
        # สร้าง tables
        print("\nCreating tables...")
        table_names = [
            'etfs', 'daily_prices', 'portfolios', 'portfolio_etfs',
            'backtests', 'backtest_transactions', 'backtest_portfolio_values', 'backtest_metrics'
        ]
        
        for i, sql in enumerate(table_sqls):
            cursor.execute(sql)
            print(f"✓ Table '{table_names[i]}' created/verified")
        
        conn.commit()
        cursor.close()
        conn.close()
        
        print("\n✓ All tables created successfully!")
        return True
        
    except Error as e:
        print(f"✗ Error creating database/tables: {e}")
        return False

# สร้าง database และ tables
if create_database_and_tables():
    print("\n🎉 Database setup complete!")
else:
    print("\n⚠️ Database setup failed!")

## Step 6: เพิ่มข้อมูล ETF ตัวอย่าง และ Portfolios

In [ ]:
def insert_sample_data():
    """เพิ่มข้อมูล ETFs และ Portfolios ตัวอย่าง"""
    
    # ข้อมูล ETFs ตัวอย่าง (40 ตัว)
    etfs_data = [
        # US Equity
        ('SPY', 'SPDR S&P 500 ETF Trust', 'US Equity', 0.0945, '1993-01-22'),
        ('VOO', 'Vanguard S&P 500 ETF', 'US Equity', 0.03, '2010-09-07'),
        ('IVV', 'iShares Core S&P 500 ETF', 'US Equity', 0.03, '2000-05-15'),
        ('QQQ', 'Invesco QQQ Trust', 'US Equity', 0.20, '1999-03-10'),
        ('VTI', 'Vanguard Total Stock Market ETF', 'US Equity', 0.03, '2001-05-24'),
        ('IWM', 'iShares Russell 2000 ETF', 'US Small Cap', 0.19, '2000-05-22'),
        ('DIA', 'SPDR Dow Jones Industrial Average ETF', 'US Equity', 0.16, '1998-01-14'),
        ('VUG', 'Vanguard Growth ETF', 'US Growth', 0.04, '2004-01-26'),
        ('VTV', 'Vanguard Value ETF', 'US Value', 0.04, '2004-01-26'),
        ('ARKK', 'ARK Innovation ETF', 'US Growth', 0.75, '2014-10-31'),
        
        # International Equity
        ('VEA', 'Vanguard FTSE Developed Markets ETF', 'International Equity', 0.05, '2007-07-20'),
        ('VWO', 'Vanguard FTSE Emerging Markets ETF', 'Emerging Markets', 0.08, '2005-03-04'),
        ('EFA', 'iShares MSCI EAFE ETF', 'International Equity', 0.32, '2001-08-14'),
        ('EEM', 'iShares MSCI Emerging Markets ETF', 'Emerging Markets', 0.68, '2003-04-07'),
        ('IEMG', 'iShares Core MSCI Emerging Markets ETF', 'Emerging Markets', 0.11, '2012-10-18'),
        
        # Bonds
        ('AGG', 'iShares Core U.S. Aggregate Bond ETF', 'Bonds', 0.03, '2003-09-22'),
        ('BND', 'Vanguard Total Bond Market ETF', 'Bonds', 0.03, '2007-04-03'),
        ('TLT', 'iShares 20+ Year Treasury Bond ETF', 'Government Bonds', 0.15, '2002-07-22'),
        ('LQD', 'iShares iBoxx Investment Grade Corporate Bond ETF', 'Corporate Bonds', 0.14, '2002-07-22'),
        ('HYG', 'iShares iBoxx High Yield Corporate Bond ETF', 'High Yield Bonds', 0.49, '2007-04-04'),
        ('BNDX', 'Vanguard Total International Bond ETF', 'International Bonds', 0.07, '2013-05-31'),
        ('TIP', 'iShares TIPS Bond ETF', 'Inflation-Protected', 0.19, '2003-12-04'),
        
        # Sector ETFs
        ('XLK', 'Technology Select Sector SPDR Fund', 'Technology', 0.10, '1998-12-16'),
        ('XLF', 'Financial Select Sector SPDR Fund', 'Financials', 0.10, '1998-12-16'),
        ('XLE', 'Energy Select Sector SPDR Fund', 'Energy', 0.10, '1998-12-16'),
        ('XLV', 'Health Care Select Sector SPDR Fund', 'Healthcare', 0.10, '1998-12-16'),
        ('XLY', 'Consumer Discretionary Select Sector SPDR', 'Consumer Discretionary', 0.10, '1998-12-16'),
        ('XLP', 'Consumer Staples Select Sector SPDR Fund', 'Consumer Staples', 0.10, '1998-12-16'),
        ('XLI', 'Industrial Select Sector SPDR Fund', 'Industrials', 0.10, '1998-12-16'),
        ('XLU', 'Utilities Select Sector SPDR Fund', 'Utilities', 0.10, '1998-12-16'),
        
        # Real Estate & Commodities
        ('VNQ', 'Vanguard Real Estate ETF', 'Real Estate', 0.12, '2004-09-23'),
        ('IYR', 'iShares U.S. Real Estate ETF', 'Real Estate', 0.41, '2000-06-12'),
        ('GLD', 'SPDR Gold Shares', 'Commodities', 0.40, '2004-11-18'),
        ('SLV', 'iShares Silver Trust', 'Commodities', 0.50, '2006-04-21'),
        ('DBC', 'Invesco DB Commodity Index Tracking Fund', 'Commodities', 0.85, '2006-02-03'),
        
        # Dividend & Income
        ('VYM', 'Vanguard High Dividend Yield ETF', 'Dividend', 0.06, '2006-11-10'),
        ('SCHD', 'Schwab U.S. Dividend Equity ETF', 'Dividend', 0.06, '2011-10-20'),
        ('DVY', 'iShares Select Dividend ETF', 'Dividend', 0.38, '2003-11-03'),
        ('SDY', 'SPDR S&P Dividend ETF', 'Dividend', 0.35, '2005-11-08'),
        ('JEPI', 'JPMorgan Equity Premium Income ETF', 'Income', 0.35, '2020-05-20')
    ]
    
    # ข้อมูล Portfolios ตัวอย่าง
    portfolios_data = [
        ('Conservative 60/40', 'Classic 60% stocks, 40% bonds portfolio for conservative investors'),
        ('Moderate 70/30', '70% stocks, 30% bonds for moderate risk tolerance'),
        ('Aggressive 90/10', '90% stocks, 10% bonds for aggressive growth'),
        ('All Weather Portfolio', 'Ray Dalio inspired diversified portfolio'),
        ('Dividend Income', 'High dividend yield portfolio for income generation'),
        ('Tech Growth', 'Technology-focused growth portfolio'),
        ('Global Diversified', 'Globally diversified portfolio across asset classes')
    ]
    
    # Portfolio allocations
    portfolio_allocations = [
        # Portfolio 1: Conservative 60/40
        [(1, 'VOO', 40.00), (1, 'VEA', 20.00), (1, 'AGG', 30.00), (1, 'BND', 10.00)],
        
        # Portfolio 2: Moderate 70/30
        [(2, 'VTI', 50.00), (2, 'VEA', 20.00), (2, 'AGG', 20.00), (2, 'TIP', 10.00)],
        
        # Portfolio 3: Aggressive 90/10
        [(3, 'VTI', 60.00), (3, 'QQQ', 20.00), (3, 'VEA', 10.00), (3, 'AGG', 10.00)],
        
        # Portfolio 4: All Weather
        [(4, 'SPY', 30.00), (4, 'TLT', 40.00), (4, 'GLD', 15.00), (4, 'DBC', 15.00)],
        
        # Portfolio 5: Dividend Income
        [(5, 'VYM', 40.00), (5, 'SCHD', 30.00), (5, 'DVY', 20.00), (5, 'JEPI', 10.00)],
        
        # Portfolio 6: Tech Growth
        [(6, 'QQQ', 50.00), (6, 'XLK', 30.00), (6, 'ARKK', 20.00)],
        
        # Portfolio 7: Global Diversified
        [(7, 'VTI', 30.00), (7, 'VEA', 20.00), (7, 'VWO', 15.00), (7, 'AGG', 20.00), (7, 'VNQ', 10.00), (7, 'GLD', 5.00)]
    ]
    
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()
        
        # ตรวจสอบว่ามีข้อมูลอยู่แล้วหรือไม่
        cursor.execute("SELECT COUNT(*) FROM etfs")
        etf_count = cursor.fetchone()[0]
        
        if etf_count > 0:
            print(f"✓ Found existing {etf_count} ETFs in database")
            cursor.execute("SELECT COUNT(*) FROM portfolios")
            portfolio_count = cursor.fetchone()[0]
            print(f"✓ Found existing {portfolio_count} portfolios in database")
            print("\nSkipping data insertion (data already exists)")
            cursor.close()
            conn.close()
            return True
        
        # Insert ETFs
        print("Inserting ETFs...")
        insert_etf_sql = "INSERT INTO etfs (ticker, name, category, expense_ratio, inception_date) VALUES (%s, %s, %s, %s, %s)"
        cursor.executemany(insert_etf_sql, etfs_data)
        print(f"✓ Inserted {len(etfs_data)} ETFs")
        
        # Insert Portfolios
        print("Inserting portfolios...")
        insert_portfolio_sql = "INSERT INTO portfolios (name, description) VALUES (%s, %s)"
        cursor.executemany(insert_portfolio_sql, portfolios_data)
        print(f"✓ Inserted {len(portfolios_data)} portfolios")
        
        # Insert Portfolio allocations
        print("Inserting portfolio allocations...")
        insert_allocation_sql = "INSERT INTO portfolio_etfs (portfolio_id, ticker, weight) VALUES (%s, %s, %s)"
        for allocations in portfolio_allocations:
            cursor.executemany(insert_allocation_sql, allocations)
        print(f"✓ Inserted portfolio allocations")
        
        conn.commit()
        cursor.close()
        conn.close()
        
        print("\n✓ Sample data inserted successfully!")
        return True
        
    except Error as e:
        print(f"✗ Error inserting sample data: {e}")
        return False

# Insert sample data
if insert_sample_data():
    print("\n🎉 Sample data ready!")
else:
    print("\n⚠️ Sample data insertion failed!")

## Step 7: ดึงข้อมูลราคา ETF จาก Yahoo Finance

⚠️ **ขั้นตอนนี้จะใช้เวลา 10-15 นาที** (ดึงข้อมูล 40 ETFs ตั้งแต่ปี 2009-2025)

คุณสามารถ:
- Run cell นี้เพื่อดึงข้อมูลเต็ม (แนะนำ)
- หรือ Skip ไปก่อนแล้วค่อยกลับมา Run ทีหลัง

In [ ]:
import yfinance as yf
from datetime import datetime, timedelta
import pandas as pd
import time

def collect_etf_prices():
    """ดึงข้อมูลราคา ETF จาก Yahoo Finance"""
    
    # ตรวจสอบว่ามีข้อมูลแล้วหรือไม่
    try:
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM daily_prices")
        price_count = cursor.fetchone()[0]
        
        if price_count > 1000:
            print(f"✓ Found existing {price_count:,} price records in database")
            print("\nSkipping data collection (data already exists)")
            print("💡 หากต้องการดึงข้อมูลใหม่ ให้ลบข้อมูลเก่าก่อน:")
            print("   DELETE FROM daily_prices;")
            cursor.close()
            conn.close()
            return True
        
        # ดึงรายการ ETFs
        cursor.execute("SELECT ticker FROM etfs ORDER BY ticker")
        tickers = [row[0] for row in cursor.fetchall()]
        cursor.close()
        conn.close()
        
        print(f"Found {len(tickers)} ETFs to collect data for")
        print(f"Date range: 2009-01-01 to {datetime.now().strftime('%Y-%m-%d')}")
        print("\nThis will take approximately 10-15 minutes...")
        print("=" * 60)
        
        start_date = '2009-01-01'
        end_date = datetime.now().strftime('%Y-%m-%d')
        
        successful = 0
        failed = []
        total = len(tickers)
        
        # ดึงข้อมูลทีละ ticker
        for i, ticker in enumerate(tickers, 1):
            try:
                # แสดง progress
                print(f"[{i}/{total}] Downloading {ticker}...", end=' ')
                
                # Download ข้อมูล (เพิ่ม auto_adjust=False เพื่อแก้ FutureWarning)
                df = yf.download(ticker, start=start_date, end=end_date, 
                               progress=False, auto_adjust=False)
                
                if df.empty:
                    print("❌ No data")
                    failed.append(ticker)
                    continue
                
                # เตรียมข้อมูลสำหรับ insert
                df.reset_index(inplace=True)
                df.columns = ['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume']
                
                # Insert เข้า database
                conn = mysql.connector.connect(**DB_CONFIG)
                cursor = conn.cursor()
                
                insert_sql = """
                INSERT INTO daily_prices (ticker, date, open, high, low, close, volume, adjusted_close)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                ON DUPLICATE KEY UPDATE
                    open = VALUES(open),
                    high = VALUES(high),
                    low = VALUES(low),
                    close = VALUES(close),
                    volume = VALUES(volume),
                    adjusted_close = VALUES(adjusted_close)
                """
                
                data = [
                    (ticker, row['date'].strftime('%Y-%m-%d'), 
                     float(row['open']), float(row['high']), float(row['low']), 
                     float(row['close']), int(row['volume']), float(row['adj_close']))
                    for _, row in df.iterrows()
                ]
                
                cursor.executemany(insert_sql, data)
                conn.commit()
                cursor.close()
                conn.close()
                
                successful += 1
                print(f"✓ ({len(df)} records)")
                
                # แสดง progress summary ทุกๆ 10 ETFs
                if i % 10 == 0:
                    print(f"\n📊 Progress: {successful}/{i} successful, {len(failed)} failed")
                    print("=" * 60)
                
            except Exception as e:
                print(f"❌ Error: {str(e)[:50]}")
                failed.append(ticker)
        
        print("\n" + "=" * 60)
        print(f"✓ Data collection complete!")
        print(f"  Successful: {successful}/{len(tickers)} ETFs")
        if failed:
            print(f"  Failed: {len(failed)} ETFs - {', '.join(failed)}")
        
        # แสดงสถิติ
        conn = mysql.connector.connect(**DB_CONFIG)
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) FROM daily_prices")
        total_records = cursor.fetchone()[0]
        cursor.close()
        conn.close()
        
        print(f"\n✓ Total price records in database: {total_records:,}")
        return True
        
    except Exception as e:
        print(f"✗ Error collecting data: {e}")
        return False

# ดึงข้อมูลราคา
print("📥 Starting data collection...\n")
if collect_etf_prices():
    print("\n🎉 Price data ready!")
else:
    print("\n⚠️ Data collection failed!")

## Step 8: Import Modules และตรวจสอบ

In [ ]:
# Import project modules
try:
    from crud_operations import crud_operations
    from backtesting import backtesting_engine
    from analytics import analytics
    import jupyter_interface as etf
    
    print("✓ crud_operations module loaded")
    print("✓ backtesting_engine module loaded")
    print("✓ analytics module loaded")
    print("✓ jupyter_interface module loaded")
    print("\n🎉 All modules ready!")
    
except Exception as e:
    print(f"✗ Error importing modules: {e}")
    print("\n💡 แนะนำ: ตรวจสอบว่าไฟล์ modules อยู่ใน directories ที่ถูกต้อง")

---

# 🎉 Setup Complete! ระบบพร้อมใช้งานแล้ว

---

## ตอนนี้คุณมี:
- ✅ Database พร้อม 8 tables
- ✅ ETFs ตัวอย่าง 40 ตัว
- ✅ Portfolios ตัวอย่าง 7 portfolios
- ✅ ข้อมูลราคาย้อนหลัง 2009-2025
- ✅ Modules พร้อมใช้งาน

---

# 📊 ตอนนี้เริ่มใช้งานได้เลย!

---

## ตัวอย่างที่ 1: ดูข้อมูล Portfolios

In [ ]:
# ดู portfolios ทั้งหมด
portfolios_df = etf.show_portfolios(DB_CONFIG)
display(portfolios_df)

## ตัวอย่างที่ 2: ดูรายละเอียด Portfolio

In [ ]:
# ดูรายละเอียด portfolio ID 1 (Conservative 60/40)
portfolio = etf.show_portfolio_details(1, DB_CONFIG)

## ตัวอย่างที่ 3: ดู ETFs ทั้งหมด

In [ ]:
# ดู ETFs ทั้งหมด
etfs_df = etf.show_etfs(DB_CONFIG)
display(etfs_df.head(10))
print(f"\nTotal ETFs: {len(etfs_df)}")

---

# 🔬 Analytics - วิเคราะห์ Portfolio

---

## Insight 1: Risk-Adjusted Performance Analysis

วิเคราะห์ความเสี่ยงและผลตอบแทนของ portfolios เทียบกับตลาด

In [ ]:
# วิเคราะห์ portfolios 1, 2, 3 (Conservative, Moderate, Aggressive)
result = etf.run_risk_analysis(
    portfolio_ids=[1, 2, 3],
    start_date='2020-01-01',
    end_date='2024-12-31',
    benchmark='SPY',
    db_config=DB_CONFIG
)

In [ ]:
# อ่าน report
with open('insight1_risk_adjusted_report.txt', 'r', encoding='utf-8') as f:
    report = f.read()
    print(report[:2000])  # แสดง 2000 ตัวอักษรแรก
    print("\n... (ดู report เต็มในไฟล์ insight1_risk_adjusted_report.txt)")

## Insight 2: Optimal Rebalancing Frequency

หาความถี่ rebalancing ที่ดีที่สุด

⚠️ **ขั้นตอนนี้จะใช้เวลา 3-5 นาที** (run 5 backtests)

In [ ]:
# วิเคราะห์ portfolio 1 (Conservative 60/40)
result = etf.run_rebalancing_analysis(
    portfolio_id=1,
    start_date='2020-01-01',
    end_date='2024-12-31',
    initial_capital=100000.0,
    db_config=DB_CONFIG
)

# แสดงผลลัพธ์สรุป
if result['success']:
    print(f"\n🏆 Best Strategy: {result['best_strategy']}")
    print(f"📈 Best Return: {result['best_return']:.2f}%")
    print(f"📊 Best Sharpe: {result['best_sharpe']:.3f}")
    print(f"\n💡 {result['recommendation']}")

In [ ]:
# อ่าน report เต็ม
with open('insight2_rebalancing_analysis.txt', 'r', encoding='utf-8') as f:
    print(f.read())

## Insight 3: DCA vs Lump Sum

เปรียบเทียบกลยุทธ์ลงทุนทีเดียว vs ทยอยซื้อ

In [ ]:
# เปรียบเทียบ DCA vs Lump Sum สำหรับ portfolio 1
result = etf.run_dca_analysis(
    portfolio_id=1,
    total_capital=100000.0,
    investment_months=12,
    start_date='2020-01-01',
    end_date='2024-12-31',
    db_config=DB_CONFIG
)

# แสดงผลลัพธ์
if result['success']:
    print(f"\n💰 Lump Sum Return: {result['lumpsum_annualized_return']:.2f}%")
    print(f"📊 DCA Return: {result['dca_annualized_return']:.2f}%")
    print(f"🏆 Winner: {result['winner']}")
    print(f"📈 DCA Win Rate: {result['dca_win_rate']:.1f}%")

In [ ]:
# อ่าน report เต็ม
with open('insight3_dca_vs_lumpsum.txt', 'r', encoding='utf-8') as f:
    print(f.read())

---

# 🎯 Run Backtest

ทดสอบกลยุทธ์การลงทุน

---

In [ ]:
# Run Buy & Hold backtest สำหรับ portfolio 1
backtest_id = etf.run_backtest(
    portfolio_id=1,
    start_date='2020-01-01',
    end_date='2024-12-31',
    initial_capital=100000.0,
    strategy='buy_hold',
    db_config=DB_CONFIG
)

print(f"\n✓ Backtest ID: {backtest_id}")

In [ ]:
# ดู backtest history
backtests_df = etf.show_backtest_history(limit=10, db_config=DB_CONFIG)
display(backtests_df)

---

# 📚 Advanced Usage

ตัวอย่างการใช้งานขั้นสูง

---

## วิเคราะห์ข้อมูลด้วย Pandas

In [ ]:
# กรอง ETFs ตาม category
etfs_df = etf.show_etfs(DB_CONFIG)

# ดูเฉพาะ US Equity
us_equity = etfs_df[etfs_df['category'] == 'US Equity']
print("US Equity ETFs:")
display(us_equity)

In [ ]:
# หา portfolios ที่มี return สูงสุด
backtests_df = etf.show_backtest_history(limit=50, db_config=DB_CONFIG)

best_backtests = backtests_df.nlargest(5, 'total_return')
print("Top 5 Best Performing Backtests:")
display(best_backtests[['portfolio_name', 'strategy_type', 'total_return', 'start_date', 'end_date']])

## Custom SQL Queries

In [ ]:
# Query ข้อมูลแบบกำหนดเอง
conn = mysql.connector.connect(**DB_CONFIG)

query = """
SELECT 
    ticker, 
    AVG(close) as avg_price, 
    MIN(close) as min_price,
    MAX(close) as max_price,
    COUNT(*) as trading_days
FROM daily_prices
WHERE date >= '2024-01-01'
GROUP BY ticker
ORDER BY avg_price DESC
LIMIT 10
"""

df = pd.read_sql(query, conn)
conn.close()

print("Top 10 ETFs by Average Price (2024):")
display(df)

---

# 🎉 สรุป

---

## คุณได้เรียนรู้:
1. ✅ Setup ระบบ ETF Backtesting ครบครัน
2. ✅ ดูข้อมูล Portfolios และ ETFs
3. ✅ Run Analytics ทั้ง 3 insights:
   - Risk-Adjusted Performance
   - Optimal Rebalancing Frequency
   - DCA vs Lump Sum
4. ✅ Run Backtests
5. ✅ วิเคราะห์ข้อมูลด้วย Pandas

---

## 📄 Reports ที่สร้าง:
- `insight1_risk_adjusted_report.txt`
- `insight2_rebalancing_analysis.txt`
- `insight3_dca_vs_lumpsum.txt`

---

## 📚 เอกสารเพิ่มเติม:
- `START_HERE_TH.md` - คู่มือเริ่มต้นจากศูนย์
- `QUICKSTART_TH.md` - Quick start 5 นาที
- `USER_GUIDE_TH.md` - คู่มือฉบับสมบูรณ์
- `ETF_Backtesting_Notebook.ipynb` - Notebook แบบละเอียด

---

## 💡 ต่อไป:
1. ลองสร้าง Portfolio ของคุณเอง
2. เปรียบเทียบกลยุทธ์ต่างๆ
3. วิเคราะห์ ETFs ที่คุณสนใจ
4. Optimize Portfolio allocation

---

# 🚀 Happy Investing! 📊

---